# 05 — Production pipeline & dashboard readiness

**Delivery ETA Intelligence** · ML engineering phase

---

### Why production ETA systems matter

Exploratory notebooks prove value; **operations** need a repeatable system: versioned features, trip-safe training, batch scoring, and monitoring hooks. This notebook wires the **production path** from processed logistics data to saved model artifacts, inference API, risk scoring, and dashboard-ready outputs.

| Capability | Business value |
|------------|----------------|
| **Real-time routing** | Score ETAs when a trip is created or rerouted |
| **Congestion-aware inference** | Graph hub features adjust OSRM-only estimates |
| **Operational deployment** | Same code in training, batch, and Streamlit |

**Inputs (API):** `source_center`, `destination_center`, `osrm_distance`, `trip_hour`, `route_type`  
**Outputs:** `predicted_eta`, `risk_level`, `bottleneck_warning`, `confidence_band`, route insights

## 1. Project architecture

```
delivery_eta/
│
├── notebooks/          # Research & validation (01–05)
├── src/                # Production Python modules
│   ├── config.py       # Paths, constants, risk thresholds
│   ├── features.py     # Feature generation
│   ├── graph_features.py
│   ├── preprocessing.py
│   ├── train.py
│   ├── inference.py
│   └── dashboard_utils.py
├── models/             # final_eta_model.pkl, graph_hub_features.pkl
├── app/                # streamlit_app.py
├── outputs/
│   ├── figures/
│   ├── tables/
│   └── predictions/
└── data/
    ├── raw/
    └── processed/
```

| Module | Responsibility |
|--------|----------------|
| `features` | Temporal, route dummies, OSRM ratios, graph interactions |
| `graph_features` | Build/load hub tables; merge source & destination |
| `preprocessing` | Fit clip/fill; transform at inference |
| `train` | Trip-level split; train XGBoost; save bundle |
| `inference` | `predict_eta()` + risk categories |
| `dashboard_utils` | Plotly charts for Streamlit |

## 2. Environment & imports

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_paths
from src.dashboard_utils import (
    eta_summary_cards,
    hub_hotspot_table,
    plot_congestion_by_lane,
    plot_eta_vs_osrm,
    plot_risk_distribution,
    save_plotly,
)
from src.features import build_ml_features, get_all_feature_names
from src.graph_features import enrich_with_graph_features, load_or_build_graph_tables
from src.inference import ETAInferencePipeline, predict_eta
from src.train import train_production_model, load_production_bundle

paths = get_paths(PROJECT_ROOT)
NOTEBOOK_TAG = "05"
paths.outputs_figures.mkdir(parents=True, exist_ok=True)
paths.outputs_predictions.mkdir(parents=True, exist_ok=True)
paths.outputs_tables.mkdir(parents=True, exist_ok=True)

def save_figure(fig, name: str):
    out = paths.outputs_figures / f"{NOTEBOOK_TAG}_{name}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out

def save_table(df, name: str):
    out = paths.outputs_tables / f"{NOTEBOOK_TAG}_{name}.csv"
    df.to_csv(out, index=False)
    return out

print("Project root:", PROJECT_ROOT)

Project root: D:\delivery eta


## 3. Train final production model

In [2]:
if not paths.final_model.exists():
    report = train_production_model(paths=paths)
else:
    bundle = load_production_bundle(paths)
    report = {"metrics": bundle["metrics"], "bundle_path": str(paths.final_model)}
    print("Loaded existing production model (delete models/final_eta_model.pkl to retrain)")

display(pd.DataFrame([report["metrics"]]))
with open(paths.model_metadata) as f:
    meta = json.load(f)
print(f"Features: {len(meta['feature_columns'])}")
print(f"Artifact: {paths.final_model}")

Loaded existing production model (delete models/final_eta_model.pkl to retrain)


,MAE,RMSE,R2,train_rows,test_rows,n_features
0,44.170184,111.940546,0.968402,113764,28738,24


Features: 24
Artifact: D:\delivery eta\models\final_eta_model.pkl


**Engineering note:** Training uses **trip-level holdout** (no segment leakage). Artifacts: `models/final_eta_model.pkl`, `models/model_metadata.json`, `models/graph_hub_features.pkl`.

## 4. Inference pipeline — single trip API

In [3]:
pipeline = ETAInferencePipeline(paths)

demo = pipeline.predict_eta(
    source_center="IND000000ACB",
    destination_center="IND562132AAA",
    osrm_distance=85.0,
    trip_hour=14,
    route_type="FTL",
)
pd.DataFrame([demo.to_dict()])

,predicted_eta,risk_level,bottleneck_warning,confidence_band,osrm_baseline,eta_vs_osrm_ratio,source_bottleneck_score,destination_bottleneck_score,route_lane,insights
0,174.394852,CRITICAL,True,"(31.231170654296875, 317.55853271484375)",64.62682,2.69849,1.0,0.507364,IND000000ACB → IND562132AAA,Touches high-bottleneck hub; expect elevated d...


### Risk scoring (operational)

| Level | Typical triggers |
|-------|------------------|
| **LOW** | Normal bottleneck scores; ETA near OSRM |
| **MEDIUM** | Elevated hub score or moderate ETA/OSRM ratio |
| **HIGH** | High bottleneck + delay ratio |
| **CRITICAL** | Top-decile bottleneck hubs and large ETA overrun |

`bottleneck_warning=True` when either hub exceeds the medium bottleneck threshold — route for capacity review.

## 5. Batch prediction simulation

In [4]:
hist = pd.read_parquet(paths.processed_parquet)

sample = hist.drop_duplicates("trip_uuid").sample(
    500,
    random_state=RANDOM_STATE
).copy()

# =========================================================
# Recreate temporal features if missing
# =========================================================

if "trip_hour" not in sample.columns:
    if "trip_creation_time" in sample.columns:
        sample["trip_creation_time"] = pd.to_datetime(sample["trip_creation_time"])
        sample["trip_hour"] = sample["trip_creation_time"].dt.hour
    else:
        sample["trip_hour"] = 12

if "trip_dayofweek" not in sample.columns:
    if "trip_creation_time" in sample.columns:
        sample["trip_dayofweek"] = sample["trip_creation_time"].dt.dayofweek
    else:
        sample["trip_dayofweek"] = 0

# =========================================================
# Create missing segment-level OSRM features
# =========================================================

if "segment_osrm_time" not in sample.columns:
    sample["segment_osrm_time"] = sample["osrm_time"]

if "segment_osrm_distance" not in sample.columns:
    sample["segment_osrm_distance"] = sample["osrm_distance"]

# =========================================================
# Build inference input
# =========================================================

required_cols = [
    "source_center",
    "destination_center",
    "osrm_distance",
    "osrm_time",
    "segment_osrm_time",
    "segment_osrm_distance",
    "trip_hour",
    "route_type",
]

batch_input = sample[required_cols].copy()

if "trip_dayofweek" in sample.columns:
    batch_input["trip_dayofweek"] = sample["trip_dayofweek"].values

# =========================================================
# Run predictions
# =========================================================

scored = pipeline.predict_batch(batch_input)

# =========================================================
# Error analysis
# =========================================================

scored["prediction_error"] = np.nan

if "actual_time" in sample.columns:
    scored["actual_time"] = sample["actual_time"].values
    scored["prediction_error"] = (
        scored["actual_time"] - scored["predicted_eta"]
    ).abs()

# =========================================================
# Save predictions
# =========================================================

out_path = paths.outputs_predictions / "sample_predictions.csv"

scored.to_csv(out_path, index=False)

print(f"Saved {len(scored)} predictions → {out_path}")

# =========================================================
# Display top risky routes
# =========================================================

display_cols = [
    "route_lane",
    "predicted_eta",
    "risk_level",
    "bottleneck_warning",
]

if "route_insights" in scored.columns:
    display_cols.append("route_insights")

top_risky = (
    scored
    .sort_values("predicted_eta", ascending=False)
    .head(10)[display_cols]
)

display(top_risky)

save_table(top_risky, "top_risky_lanes")

Saved 500 predictions → D:\delivery eta\outputs\predictions\sample_predictions.csv


,route_lane,predicted_eta,risk_level,bottleneck_warning,route_insights
311,IND792121AAB → IND786181AAC,637.191711,MEDIUM,False,Predicted ETA materially above OSRM baseline.
208,IND796009AAA → IND796321AAB,191.423645,MEDIUM,False,Predicted ETA materially above OSRM baseline.
118,IND756100AAC → IND758034AAA,138.614120,MEDIUM,False,Predicted ETA materially above OSRM baseline.
19,IND713205AAB → IND722101AAB,120.453743,MEDIUM,False,Predicted ETA materially above OSRM baseline.
276,IND786181AAC → IND792120AAB,117.769196,MEDIUM,False,Predicted ETA materially above OSRM baseline.
69,IND263601AAA → IND262501AAA,105.739250,LOW,False,Predicted ETA materially above OSRM baseline.
444,IND411033AAA → IND411014AAA,99.211357,MEDIUM,False,Predicted ETA materially above OSRM baseline.
80,IND624601AAA → IND624101AAA,96.988174,LOW,False,Within normal network operating envelope.
377,IND712311AAA → IND742101AAC,96.433975,MEDIUM,False,Predicted ETA materially above OSRM baseline.
341,IND400015AAB → IND400072AAB,92.682083,MEDIUM,False,Predicted ETA materially above OSRM baseline.


WindowsPath('D:/delivery eta/outputs/tables/05_top_risky_lanes.csv')

## 6. Dashboard-ready visualizations

In [5]:
cards = eta_summary_cards(scored)
for k, v in cards.items():
    print(f"{k}: {v}")

fig1 = plot_risk_distribution(scored)
save_plotly(fig1, paths.outputs_figures / f"{NOTEBOOK_TAG}_risk_distribution")

fig2 = plot_eta_vs_osrm(scored)
save_plotly(fig2, paths.outputs_figures / f"{NOTEBOOK_TAG}_eta_vs_osrm")

fig3 = plot_congestion_by_lane(scored)
save_plotly(fig3, paths.outputs_figures / f"{NOTEBOOK_TAG}_congestion_lanes")

hotspots = hub_hotspot_table(scored)
display(hotspots.head(15))
save_table(hotspots, "hub_hotspots")

# Matplotlib summary for static reports
fig, ax = plt.subplots(figsize=(8, 4))
scored["risk_level"].value_counts().reindex(["LOW", "MEDIUM", "HIGH", "CRITICAL"]).plot.bar(ax=ax, color="steelblue")
ax.set_title("Risk distribution (batch sample)")
save_figure(fig, "risk_bar")

trips_scored: 500
mean_predicted_eta: 38.08345413208008
critical_risk_count: 40
bottleneck_alerts: 107


,hub_id,risky_trips
0,IND000000ACB,62.0
35,IND562132AAA,21.0
15,IND131028AAB,8.0
7,IND110044AAB,6.0
34,IND560300AAA,6.0
9,IND110064AAA,5.0
1,IND000000ACT,4.0
30,IND560058AAD,4.0
23,IND382430AAB,3.0
33,IND560099AAB,3.0


WindowsPath('D:/delivery eta/outputs/figures/05_risk_bar.png')

**Dashboard:** Run `streamlit run app/streamlit_app.py` — consumes `sample_predictions.csv` and live `predict_eta()` inputs.

HTML Plotly exports saved under `outputs/figures/05_*.html` for embedding without a server.

## 7. Model monitoring concepts

| Monitor | Definition | Action |
|---------|------------|--------|
| **Data drift** | Input distribution shift (distance, hour, route mix) | Alert + review retrain |
| **Concept drift** | MAE ↑ while inputs stable | Refresh model / graph tables |
| **SLA degradation** | % trips CRITICAL risk ↑ | Ops playbook on bottleneck hubs |
| **Feature freshness** | Graph tables stale > 7d | Nightly `build_lane_graph_tables` job |
| **Retraining** | Rolling 90d window monthly | Champion/challenger via trip holdout |

Log every inference: inputs, prediction, risk, model version, graph snapshot date.

## 8. Production & deployment architecture

```
[Trip event] → [Feature store / router OSRM]
        ↓
[Inference service: predict_eta()]
        ↓
[Risk engine] → [Dashboard / SLA alerts]
        ↓
[Kafka sink] → [Monitoring & retrain pipeline]
```

- **API serving:** FastAPI wrapper around `ETAInferencePipeline` (single worker loads bundle at startup).
- **Streaming:** Kafka topic `trip.created` → stream processor joins graph features from Redis/feature store.
- **Graph refresh:** Nightly batch on processed segments; publish `graph_hub_features.pkl` version.
- **Real-time inference:** p99 target < 50ms with preloaded model + in-memory hub dict lookup.

## 9. Executive summary

In [6]:
metrics = report["metrics"]
print("=" * 60)
print("PRODUCTION ETA SYSTEM — PHASE 5 SUMMARY")
print("=" * 60)
print(f"Model: XGBoost @ {paths.final_model.name}")
print(f"Test MAE: {metrics.get('MAE', 'n/a'):.2f} min | R²: {metrics.get('R2', 0):.4f}")
print(f"Batch predictions: {paths.outputs_predictions / 'sample_predictions.csv'}")
print(f"Streamlit: streamlit run app/streamlit_app.py")
print("Graph features: embedded in bundle (PageRank, bottleneck, community)")
print("Next: Node2Vec embeddings · temporal graph · traffic API · streaming deploy")

PRODUCTION ETA SYSTEM — PHASE 5 SUMMARY
Model: XGBoost @ final_eta_model.pkl
Test MAE: 44.17 min | R²: 0.9684
Batch predictions: D:\delivery eta\outputs\predictions\sample_predictions.csv
Streamlit: streamlit run app/streamlit_app.py
Graph features: embedded in bundle (PageRank, bottleneck, community)
Next: Node2Vec embeddings · temporal graph · traffic API · streaming deploy
